In [1]:
pip install rank_bm25 sentence-transformers tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
os.chdir(r"c:\Users\STUDENT\Downloads\BioASQ-training14b (1)\BioASQ-training14b")
print(os.listdir())

['README', 'training14b.json']


In [ ]:
# For the full dataset which will serve as benchmark, we will use a simple BM25 retrieval method to establish a baseline. 
# This will allow us to evaluate the difficulty of the questions and the quality of the snippets provided.

import json
import numpy as np
from rank_bm25 import BM25Okapi
from tqdm import tqdm
import re

# ── Load Data ──────────────────────────────────────────────────────────────────
with open("training14b.json", "r", encoding="utf-8") as f:
    data = json.load(f)

questions = data["questions"]
print(f"Loaded {len(questions)} questions")

# # ── Sample 1000 questions ──────────────────────────────────────────────────────
# import random
# random.seed(42)
# questions = random.sample(questions, 1000)
# print(f"Sampled 1000 questions")

# ── Helpers ────────────────────────────────────────────────────────────────────
def tokenize(text):
    return re.findall(r'\w+', text.lower())

def average_precision(relevant_docs, retrieved_docs):
    """Compute Average Precision for one question."""
    if not relevant_docs:
        return 0.0
    relevant_set = set(relevant_docs)
    hits = 0
    sum_precision = 0.0
    for i, doc in enumerate(retrieved_docs):
        if doc in relevant_set:
            hits += 1
            sum_precision += hits / (i + 1)
    return sum_precision / len(relevant_set)

# ── Build Global Snippet Corpus ────────────────────────────────────────────────
# Each entry = (snippet_text_tokens, document_url)
print("Building corpus...")
corpus = []
for q in questions:
    for snippet in q.get("snippets", []):
        corpus.append({
            "tokens": tokenize(snippet["text"]),
            "text": snippet["text"],
            "document": snippet["document"]
        })

print(f"Total snippets in corpus: {len(corpus)}")

# Build BM25 index
bm25 = BM25Okapi([entry["tokens"] for entry in corpus])

# ── Evaluate ───────────────────────────────────────────────────────────────────
TOP_K = 10  # How many documents to retrieve per question
ap_scores = []

print("Running BM25 retrieval...")
for q in tqdm(questions):
    query_tokens = tokenize(q["body"])
    ground_truth_docs = q.get("documents", [])

    if not ground_truth_docs or not q.get("snippets"):
        continue

    # Score all snippets
    scores = bm25.get_scores(query_tokens)

    # Rank snippets by score
    ranked_indices = np.argsort(scores)[::-1]

    # Deduplicate: get top-K unique document URLs
    retrieved_docs = []
    seen = set()
    for idx in ranked_indices:
        doc_url = corpus[idx]["document"]
        if doc_url not in seen:
            seen.add(doc_url)
            retrieved_docs.append(doc_url)
        if len(retrieved_docs) >= TOP_K:
            break

    ap = average_precision(ground_truth_docs, retrieved_docs)
    ap_scores.append(ap)

# ── Results ────────────────────────────────────────────────────────────────────
map_score = np.mean(ap_scores)
print(f"\n=== BM25 Baseline Results ===")
print(f"Questions evaluated: {len(ap_scores)}")
print(f"MAP @ {TOP_K}: {map_score:.4f}")

# Breakdown by question type
for qtype in ["yesno", "factoid", "list", "summary"]:
    type_scores = []
    for q, ap in zip(questions, ap_scores):
        if q.get("type") == qtype:
            type_scores.append(ap)
    if type_scores:
        print(f"  MAP ({qtype}): {np.mean(type_scores):.4f}  ({len(type_scores)} questions)")

In [3]:
# Note: This is a very basic BM25 baseline. It does not use any of the snippet texts for retrieval, only the document URLs. 
# More sophisticated methods could incorporate snippet relevance, use embeddings, or apply re-ranking to improve performance.

import json
import numpy as np
from rank_bm25 import BM25Okapi
from tqdm import tqdm
import re

# ── Load Data ──────────────────────────────────────────────────────────────────
with open("training14b.json", "r", encoding="utf-8") as f:
    data = json.load(f)

questions = data["questions"]
print(f"Loaded {len(questions)} questions")

# ── Sample 1000 questions ──────────────────────────────────────────────────────
import random
random.seed(42)
questions = random.sample(questions, 1000)
print(f"Sampled 1000 questions")

# ── Helpers ────────────────────────────────────────────────────────────────────
def tokenize(text):
    return re.findall(r'\w+', text.lower())

def average_precision(relevant_docs, retrieved_docs):
    """Compute Average Precision for one question."""
    if not relevant_docs:
        return 0.0
    relevant_set = set(relevant_docs)
    hits = 0
    sum_precision = 0.0
    for i, doc in enumerate(retrieved_docs):
        if doc in relevant_set:
            hits += 1
            sum_precision += hits / (i + 1)
    return sum_precision / len(relevant_set)

# ── Build Global Snippet Corpus ────────────────────────────────────────────────
# Each entry = (snippet_text_tokens, document_url)
print("Building corpus...")
corpus = []
for q in questions:
    for snippet in q.get("snippets", []):
        corpus.append({
            "tokens": tokenize(snippet["text"]),
            "text": snippet["text"],
            "document": snippet["document"]
        })

print(f"Total snippets in corpus: {len(corpus)}")

# Build BM25 index
bm25 = BM25Okapi([entry["tokens"] for entry in corpus])

# ── Evaluate ───────────────────────────────────────────────────────────────────
TOP_K = 10  # How many documents to retrieve per question
ap_scores = []

print("Running BM25 retrieval...")
for q in tqdm(questions):
    query_tokens = tokenize(q["body"])
    ground_truth_docs = q.get("documents", [])

    if not ground_truth_docs or not q.get("snippets"):
        continue

    # Score all snippets
    scores = bm25.get_scores(query_tokens)

    # Rank snippets by score
    ranked_indices = np.argsort(scores)[::-1]

    # Deduplicate: get top-K unique document URLs
    retrieved_docs = []
    seen = set()
    for idx in ranked_indices:
        doc_url = corpus[idx]["document"]
        if doc_url not in seen:
            seen.add(doc_url)
            retrieved_docs.append(doc_url)
        if len(retrieved_docs) >= TOP_K:
            break

    ap = average_precision(ground_truth_docs, retrieved_docs)
    ap_scores.append(ap)

# ── Results ────────────────────────────────────────────────────────────────────
map_score = np.mean(ap_scores)
print(f"\n=== BM25 Baseline Results ===")
print(f"Questions evaluated: {len(ap_scores)}")
print(f"MAP @ {TOP_K}: {map_score:.4f}")

# Breakdown by question type
for qtype in ["yesno", "factoid", "list", "summary"]:
    type_scores = []
    for q, ap in zip(questions, ap_scores):
        if q.get("type") == qtype:
            type_scores.append(ap)
    if type_scores:
        print(f"  MAP ({qtype}): {np.mean(type_scores):.4f}  ({len(type_scores)} questions)")

Loaded 5729 questions
Sampled 1000 questions
Building corpus...
Total snippets in corpus: 14554
Running BM25 retrieval...


100%|██████████| 1000/1000 [00:36<00:00, 27.29it/s]


=== BM25 Baseline Results ===
Questions evaluated: 1000
MAP @ 10: 0.5262
  MAP (yesno): 0.5790  (256 questions)
  MAP (factoid): 0.5417  (302 questions)
  MAP (list): 0.4136  (195 questions)
  MAP (summary): 0.5412  (247 questions)


In [4]:
import torch
print(torch.cuda.is_available())

False


In [5]:
import psutil
ram = psutil.virtual_memory()
print(f"Total RAM: {ram.total / (1024**3):.1f} GB")
print(f"Available RAM: {ram.available / (1024**3):.1f} GB")

Total RAM: 15.6 GB
Available RAM: 1.4 GB


In [6]:
pip install sentence-transformers


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import os
os.chdir(r"c:\Users\STUDENT\Downloads\BioASQ-training14b (1)\BioASQ-training14b")

import json
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import re
import random

# ── Load Data ──────────────────────────────────────────────────────────────────
with open("training14b.json", "r", encoding="utf-8") as f:
    data = json.load(f)

questions = data["questions"]
print(f"Loaded {len(questions)} questions")

random.seed(42)
questions = random.sample(questions, 1000)
print(f"Sampled 1000 questions")

# ── Helpers ────────────────────────────────────────────────────────────────────
def tokenize(text):
    return re.findall(r'\w+', text.lower())

def average_precision(relevant_docs, retrieved_docs):
    if not relevant_docs:
        return 0.0
    relevant_set = set(relevant_docs)
    hits = 0
    sum_precision = 0.0
    for i, doc in enumerate(retrieved_docs):
        if doc in relevant_set:
            hits += 1
            sum_precision += hits / (i + 1)
    return sum_precision / len(relevant_set)

def reciprocal_rank_fusion(bm25_docs, dense_docs, k=60):
    """Combine two ranked lists using RRF."""
    scores = {}
    for rank, doc in enumerate(bm25_docs):
        scores[doc] = scores.get(doc, 0) + 1 / (k + rank + 1)
    for rank, doc in enumerate(dense_docs):
        scores[doc] = scores.get(doc, 0) + 1 / (k + rank + 1)
    return sorted(scores.keys(), key=lambda x: scores[x], reverse=True)

# ── Build Corpus ───────────────────────────────────────────────────────────────
print("Building corpus...")
corpus = []
for q in questions:
    for snippet in q.get("snippets", []):
        corpus.append({
            "tokens": tokenize(snippet["text"]),
            "text": snippet["text"],
            "document": snippet["document"]
        })

print(f"Total snippets in corpus: {len(corpus)}")

# ── BM25 Index ─────────────────────────────────────────────────────────────────
print("Building BM25 index...")
bm25 = BM25Okapi([entry["tokens"] for entry in corpus])

# ── Dense Embeddings ───────────────────────────────────────────────────────────
print("Loading embedding model (bge-small)...")
model = SentenceTransformer('BAAI/bge-small-en-v1.5')

print("Encoding corpus snippets in batches (this will take a few minutes)...")
batch_size = 32
corpus_texts = [entry["text"] for entry in corpus]
corpus_embeddings = []

for i in tqdm(range(0, len(corpus_texts), batch_size)):
    batch = corpus_texts[i:i+batch_size]
    embeddings = model.encode(batch, normalize_embeddings=True)
    corpus_embeddings.append(embeddings)

corpus_embeddings = np.vstack(corpus_embeddings)
print(f"Encoded {len(corpus_embeddings)} snippets")

# ── Evaluate Hybrid ────────────────────────────────────────────────────────────
TOP_K = 10
ap_scores = []

print("Running Hybrid BM25 + Dense + RRF retrieval...")
for q in tqdm(questions):
    query_tokens = tokenize(q["body"])
    ground_truth_docs = q.get("documents", [])

    if not ground_truth_docs or not q.get("snippets"):
        continue

    # BM25 ranking
    bm25_scores = bm25.get_scores(query_tokens)
    bm25_ranked = np.argsort(bm25_scores)[::-1]
    bm25_docs = []
    seen = set()
    for idx in bm25_ranked:
        doc = corpus[idx]["document"]
        if doc not in seen:
            seen.add(doc)
            bm25_docs.append(doc)
        if len(bm25_docs) >= 50:
            break

    # Dense ranking
    query_embedding = model.encode([q["body"]], normalize_embeddings=True)
    dense_scores = np.dot(corpus_embeddings, query_embedding.T).squeeze()
    dense_ranked = np.argsort(dense_scores)[::-1]
    dense_docs = []
    seen = set()
    for idx in dense_ranked:
        doc = corpus[idx]["document"]
        if doc not in seen:
            seen.add(doc)
            dense_docs.append(doc)
        if len(dense_docs) >= 50:
            break

    # RRF fusion
    fused_docs = reciprocal_rank_fusion(bm25_docs, dense_docs)[:TOP_K]

    ap = average_precision(ground_truth_docs, fused_docs)
    ap_scores.append(ap)

# ── Results ────────────────────────────────────────────────────────────────────
map_score = np.mean(ap_scores)
print(f"\n=== Hybrid BM25 + Dense (RRF) Results ===")
print(f"Questions evaluated: {len(ap_scores)}")
print(f"MAP @ {TOP_K}: {map_score:.4f}")

for qtype in ["yesno", "factoid", "list", "summary"]:
    type_scores = []
    for q, ap in zip(questions, ap_scores):
        if q.get("type") == qtype:
            type_scores.append(ap)
    if type_scores:
        print(f"  MAP ({qtype}): {np.mean(type_scores):.4f}  ({len(type_scores)} questions)")

Loaded 5729 questions
Sampled 1000 questions
Building corpus...
Total snippets in corpus: 14554
Building BM25 index...
Loading embedding model (bge-small)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\STUDENT\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\STUDENT\.cache\huggingface\hub\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

Encoding corpus snippets in batches (this will take a few minutes)...


100%|██████████| 455/455 [16:05<00:00,  2.12s/it]


Encoded 14554 snippets
Running Hybrid BM25 + Dense + RRF retrieval...


100%|██████████| 1000/1000 [03:33<00:00,  4.68it/s]



=== Hybrid BM25 + Dense (RRF) Results ===
Questions evaluated: 1000
MAP @ 10: 0.6343
  MAP (yesno): 0.6548  (256 questions)
  MAP (factoid): 0.6563  (302 questions)
  MAP (list): 0.5478  (195 questions)
  MAP (summary): 0.6543  (247 questions)
